In [14]:
import json
from pathlib import Path

import pandas as pd

from sentence_transformers import SentenceTransformer

In [15]:
CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name == "notebooks":
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR


DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "outputs"


train_poc = pd.read_csv(
    DATA_DIR / "train_poc.csv"
)

test_poc = pd.read_csv(
    DATA_DIR / "test_poc.csv"
)


stage_a_train = pd.read_csv(
    OUTPUT_DIR / "stage_a_train_predictions.csv"
)

stage_a_test = pd.read_csv(
    OUTPUT_DIR / "stage_a_test_predictions.csv"
)


print("Train records:", len(train_poc))
print("Test records:", len(test_poc))
print("Stage A train predictions:", len(stage_a_train))
print("Stage A test predictions:", len(stage_a_test))

Train records: 3322
Test records: 819
Stage A train predictions: 3322
Stage A test predictions: 819


In [16]:
# VALIDATION CELL
# if not train_poc["id"].isin(
#     stage_a_train["id"]
# ).all():

#     raise ValueError(
#         "Some training records do not have Stage A predictions."
#     )


# if not test_poc["id"].isin(
#     stage_a_test["id"]
# ).all():

#     raise ValueError(
#         "Some test records do not have Stage A predictions."
#     )


# print(
#     "Stage A prediction IDs match the POC datasets."
# )

In [17]:
# CSV stores prediction lists as text.
stage_a_train["stage_a_predictions"] = (
    stage_a_train["stage_a_predictions"].apply(
        json.loads
    )
)

stage_a_test["stage_a_predictions"] = (
    stage_a_test["stage_a_predictions"].apply(
        json.loads
    )
)

In [18]:
# validation cell
# Confirm that every POC record has a Stage A result.
if not train_poc["id"].isin(
    stage_a_train["id"]
).all():
    raise ValueError(
        "Some training records do not have Stage A predictions."
    )


if not test_poc["id"].isin(
    stage_a_test["id"]
).all():
    raise ValueError(
        "Some test records do not have Stage A predictions."
    )


print(
    "Stage A prediction IDs match the POC datasets."
)

Stage A prediction IDs match the POC datasets.


In [19]:
# Define the risk descriptions
RISK_TAXONOMY = {
    "weather_disruption": {
        "name": "Weather Disruption",
        "description": (
            "Disruption to trade, transport, ports, logistics, or infrastructure "
            "caused by severe weather such as storms, flooding, high winds, "
            "cyclones, or similar weather-related hazards."
        )
    },

    "natural_disaster": {
        "name": "Natural Disaster",
        "description": (
            "Disruption caused by natural disasters such as earthquakes, "
            "tsunamis, volcanic activity, landslides, or other geological hazards."
        )
    },

    "port_operational_disruption": {
        "name": "Port Operational Disruption",
        "description": (
            "Disruption to normal port or cargo operations caused by congestion, "
            "capacity limitations, operational delays, cargo disruption, "
            "or reduced terminal efficiency."
        )
    },

    "port_closure": {
        "name": "Port Closure",
        "description": (
            "Full or partial closure, suspension, or shutdown of a port, terminal, "
            "pier, berth, or related maritime facility."
        )
    },

    "labor_strike_disruption": {
        "name": "Labor / Strike Disruption",
        "description": (
            "Disruption caused by worker strikes, industrial action, labor disputes, "
            "walkouts, or related workforce actions affecting transport or logistics."
        )
    },

    "maritime_security_navigation_disruption": {
        "name": "Maritime Security / Navigation Disruption",
        "description": (
            "Disruption or increased operational risk to maritime transport caused "
            "by maritime advisories, piracy, security threats, navigation restrictions, "
            "or waterway closures and disruptions."
        )
    }
}

In [20]:
# Load the semantic embedding model
MODEL_NAME = "all-MiniLM-L6-v2"

semantic_model = SentenceTransformer(
    MODEL_NAME
)

print(
    "Semantic model loaded:",
    MODEL_NAME
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Semantic model loaded: all-MiniLM-L6-v2


In [21]:
# Create embeddings for the risk descriptions

risk_ids = list(
    RISK_TAXONOMY.keys()
)

risk_texts = [
    (
        RISK_TAXONOMY[risk_id]["name"]
        + ". "
        + RISK_TAXONOMY[risk_id]["description"]
    )
    for risk_id in risk_ids
]


risk_embeddings = semantic_model.encode(
    risk_texts,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print(
    "Risk embeddings shape:",
    risk_embeddings.shape
)

Risk embeddings shape: (6, 384)


In [22]:
# Create embeddings for all train and test articles
train_texts = (
    train_poc["input_text"]
    .fillna("")
    .astype(str)
    .tolist()
)

test_texts = (
    test_poc["input_text"]
    .fillna("")
    .astype(str)
    .tolist()
)


train_embeddings = semantic_model.encode(
    train_texts,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

test_embeddings = semantic_model.encode(
    test_texts,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)


print(
    "Train embeddings shape:",
    train_embeddings.shape
)

print(
    "Test embeddings shape:",
    test_embeddings.shape
)

Batches:   0%|          | 0/104 [00:00<?, ?it/s]

Batches:   0%|          | 0/26 [00:00<?, ?it/s]

Train embeddings shape: (3322, 384)
Test embeddings shape: (819, 384)


In [23]:
# Calculate semantic similarity scores

# Because all embeddings are normalized, the dot product
# gives the cosine similarity between articles and risks.
train_semantic_scores = (
    train_embeddings
    @ risk_embeddings.T
)

test_semantic_scores = (
    test_embeddings
    @ risk_embeddings.T
)


print(
    "Train semantic scores shape:",
    train_semantic_scores.shape
)

print(
    "Test semantic scores shape:",
    test_semantic_scores.shape
)


Train semantic scores shape: (3322, 6)
Test semantic scores shape: (819, 6)


In [24]:
#Convert semantic scores into predictions

SEMANTIC_THRESHOLD = 0.35


def get_semantic_predictions(
    semantic_scores,
    threshold
):
    all_predictions = []

    for row_scores in semantic_scores:
        row_predictions = [
            risk_id
            for risk_id, score in zip(
                risk_ids,
                row_scores
            )
            if score >= threshold
        ]

        # Ensure that every POC article receives
        # at least one semantic prediction.
        if not row_predictions:
            best_risk_index = row_scores.argmax()

            row_predictions = [
                risk_ids[best_risk_index]
            ]

        all_predictions.append(
            row_predictions
        )

    return all_predictions

In [25]:
# Classify the train and test data

train_poc["stage_b_predictions"] = (
    get_semantic_predictions(
        train_semantic_scores,
        SEMANTIC_THRESHOLD
    )
)

test_poc["stage_b_predictions"] = (
    get_semantic_predictions(
        test_semantic_scores,
        SEMANTIC_THRESHOLD
    )
)


print(
    "Stage B training classifications:",
    len(train_poc["stage_b_predictions"])
)

print(
    "Stage B test classifications:",
    len(test_poc["stage_b_predictions"])
)

Stage B training classifications: 3322
Stage B test classifications: 819


In [26]:
# Inspect the semantic classifications

train_poc[
    [
        "id",
        "Headline",
        "stage_b_predictions"
    ]
].head(10)

,id,Headline,stage_b_predictions
0,552,Severe winds caused brief suspension at Port o...,"[port_operational_disruption, port_closure]"
1,2973,USA: 'Fridays for Future' climate change prote...,[weather_disruption]
2,6,UPDATE - Indonesia: Severe winds damage infras...,[weather_disruption]
3,4537,UPDATE 1 - Refrigerated container import capac...,[port_operational_disruption]
4,3771,Marine wind warning issued for Port of Sydney ...,"[weather_disruption, maritime_security_navigat..."
5,3090,WATCH FOR: Road cargo and transport disruption...,[labor_strike_disruption]
6,1947,Osaka’s G20 summit likely to impede logistics ...,[maritime_security_navigation_disruption]
7,914,UPDATE: Up to 11 vessels waiting for a berth a...,[port_closure]
8,3927,Port congestion reported at Port of Dammam due...,"[port_operational_disruption, port_closure]"
9,809,"UPDATE - USA, Virginia: Tropical Storm Michael...",[weather_disruption]


In [28]:
# Prepare ground-truth labels
import ast
import numpy as np

from sklearn.metrics import classification_report
from sklearn.metrics import precision_recall_fscore_support
from sklearn.preprocessing import MultiLabelBinarizer


# Restore the true risk lists loaded from CSV.
def restore_list(value):
    if isinstance(value, list):
        return value

    if pd.isna(value):
        return []

    return ast.literal_eval(value)


train_poc["true_risks"] = (
    train_poc["true_risks"].apply(
        restore_list
    )
)

test_poc["true_risks"] = (
    test_poc["true_risks"].apply(
        restore_list
    )
)

In [29]:
# Define the Stage B evaluation function

RISK_IDS = list(
    RISK_TAXONOMY.keys()
)


def evaluate_stage_b(dataframe, split_name):

    label_binarizer = MultiLabelBinarizer(
        classes=RISK_IDS
    )

    true_matrix = label_binarizer.fit_transform(
        dataframe["true_risks"]
    )

    predicted_matrix = label_binarizer.transform(
        dataframe["stage_b_predictions"]
    )

    print(
        f"{split_name} Stage B classification report"
    )

    print(
        classification_report(
            true_matrix,
            predicted_matrix,
            target_names=[
                RISK_TAXONOMY[risk_id]["name"]
                for risk_id in RISK_IDS
            ],
            zero_division=0
        )
    )

    micro_precision, micro_recall, micro_f1, _ = (
        precision_recall_fscore_support(
            true_matrix,
            predicted_matrix,
            average="micro",
            zero_division=0
        )
    )

    macro_precision, macro_recall, macro_f1, _ = (
        precision_recall_fscore_support(
            true_matrix,
            predicted_matrix,
            average="macro",
            zero_division=0
        )
    )

    exact_match = np.mean(
        [
            set(true_risks) == set(predicted_risks)
            for true_risks, predicted_risks in zip(
                dataframe["true_risks"],
                dataframe["stage_b_predictions"]
            )
        ]
    )

    print(
        "Micro precision:",
        round(micro_precision, 4)
    )

    print(
        "Micro recall:",
        round(micro_recall, 4)
    )

    print(
        "Micro F1:",
        round(micro_f1, 4)
    )

    print(
        "Macro precision:",
        round(macro_precision, 4)
    )

    print(
        "Macro recall:",
        round(macro_recall, 4)
    )

    print(
        "Macro F1:",
        round(macro_f1, 4)
    )

    print(
        "Exact multi-label match:",
        round(exact_match, 4)
    )

    return {
        "micro_precision": micro_precision,
        "micro_recall": micro_recall,
        "micro_f1": micro_f1,
        "macro_precision": macro_precision,
        "macro_recall": macro_recall,
        "macro_f1": macro_f1,
        "exact_match": exact_match
    }

In [30]:
# Evaluate training predictions

train_stage_b_metrics = evaluate_stage_b(
    train_poc,
    "Training"
)

Training Stage B classification report
                                           precision    recall  f1-score   support

                       Weather Disruption       0.92      0.53      0.68      1300
                         Natural Disaster       0.64      0.87      0.74       103
              Port Operational Disruption       0.66      0.69      0.67      1394
                             Port Closure       0.27      0.82      0.40       355
                Labor / Strike Disruption       0.96      0.75      0.84       740
Maritime Security / Navigation Disruption       0.31      0.39      0.34       464

                                micro avg       0.60      0.63      0.62      4356
                                macro avg       0.63      0.67      0.61      4356
                             weighted avg       0.72      0.63      0.65      4356
                              samples avg       0.65      0.67      0.63      4356

Micro precision: 0.6029
Micro recall: 0.6348


In [31]:
# Evaluate test predictions

test_stage_b_metrics = evaluate_stage_b(
    test_poc,
    "Test"
)

Test Stage B classification report
                                           precision    recall  f1-score   support

                       Weather Disruption       0.94      0.53      0.68       341
                         Natural Disaster       0.69      0.89      0.78        35
              Port Operational Disruption       0.68      0.68      0.68       351
                             Port Closure       0.27      0.82      0.41        85
                Labor / Strike Disruption       0.95      0.79      0.86       179
Maritime Security / Navigation Disruption       0.30      0.43      0.35        97

                                micro avg       0.62      0.65      0.63      1088
                                macro avg       0.64      0.69      0.63      1088
                             weighted avg       0.74      0.65      0.66      1088
                              samples avg       0.66      0.68      0.65      1088

Micro precision: 0.6167
Micro recall: 0.6461
Micr

In [32]:
# Compare the two reports
stage_b_metric_comparison = pd.DataFrame(
    [
        train_stage_b_metrics,
        test_stage_b_metrics
    ],
    index=[
        "Training",
        "Test"
    ]
)

stage_b_metric_comparison.round(4)

,micro_precision,micro_recall,micro_f1,macro_precision,macro_recall,macro_f1,exact_match
Training,0.6029,0.6348,0.6184,0.6263,0.6748,0.6126,0.4266
Test,0.6167,0.6461,0.6311,0.6359,0.6903,0.6250,0.4396
